## SECTION 2.2 텍스트 토큰화하기

In [2]:
import urllib.request

url = ("https://raw.githubusercontent.com/rickiepark/"
       "llm-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x15d09d41600>)

In [3]:
# 총 문자 수, 처음 99개 문자 출력
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()
print("총 문자 개수:", len(raw_text))
print(raw_text[:99])

총 문자 개수: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [4]:
# 개별 단어, 공백, 구두점 문자로 이루어진 리스트 출력
import re
text = 'Hello, world. This, is a test.'
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [5]:
# 특수 문자 처리
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip () for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [6]:
# 구현한 토크나이저 이용해 소설에 적용
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip () for item in preprocessed if item.strip()]
print(len(preprocessed))

print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## SECTION 2.3 토큰을 토큰 ID로 변환하기

In [7]:
# 모든 고유 토큰의 리스트 만들고 알파벳 순으로 정렬, 어휘 사전 크기 확인
# 어휘 사전의 크기: 1130
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [8]:
# 단어 사전의 첫 50개 출력
vocab = {token:integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
  print(item)
  if i >= 50:
    break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [9]:
# 간단한 텍스트 토크나이저 구현

class SimpleTokenizerV1:
  def __init__(self, vocab):
    self.str_to_int = vocab     # encode 메서드와 decode 메서드에 참조할 수 있도록 어휘 사전을 클래스의 속성으로 지정
    self.int_to_str = {i:s for s, i in vocab.items()}   # 토큰 ID를 원본 텍스트 토큰으로 매핑하는 역어휘 사전 생성

  # 입력 텍스트를 처리해 토큰 ID로 변경
  def encode(self, text):
      preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
      preprocessed = [
          item.strip() for item in preprocessed if item.strip()
      ]
      ids = [self.str_to_int[s] for s in preprocessed]
      return ids

  # 토큰 ID를 텍스트로 변경
  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) # 지정된 구두점 문자 앞의 공백 삭제
    return text

In [10]:
# 토크나이저 간단한 테스트
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
  Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [11]:
# 토큰 ID를 다시 텍스트로 변환
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [12]:
# # 훈련 세트에 없는 새로운 텍스트 샘플 적용
# text = "Hello, do you like tea?"
# print(tokenizer.encode(text))

# # KeyError 발생

## SECTION 2.4 특수 문맥 토큰 추가하기

- **`<|unk|>`**: 훈련 데이터에 포함되어 있지 않아 기존의 어휘 사전에 없는 새로운 단어를 나타냄
- **`<|endoftext>`**: 관련이 없는 2개의 텍스트를 구분하는 데 사용

In [13]:
# 특수 토큰 <|unk|>, <|endoftext|> 추가
# 결과가 1130 -> 1132 됨
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer, token in enumerate(all_tokens)}

print(len(vocab.items()))

print()

# 어휘 사전의 마지막 5개 항목 출력
for i, item in enumerate(list(vocab.items())[-5:]):
  print(item)

1132

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [14]:
# 알지 못하는 단어를 처리하는 토크나이저 구현

class SimpleTokenizerV2:
  def __init__(self, vocab):
    self.str_to_int = vocab     # encode 메서드와 decode 메서드에 참조할 수 있도록 어휘 사전을 클래스의 속성으로 지정
    self.int_to_str = {i:s for s, i in vocab.items()}   # 토큰 ID를 원본 텍스트 토큰으로 매핑하는 역어휘 사전 생성

  # 입력 텍스트를 처리해 토큰 ID로 변경
  def encode(self, text):
      preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
      preprocessed = [
          item.strip() for item in preprocessed if item.strip()
      ]
      preprocessed = [item if item in self.str_to_int   # 알지 못하는 단어를 <|unk|> 토큰으로 변환
                      else "<|unk|>" for item in preprocessed]
      ids = [self.str_to_int[s] for s in preprocessed]
      return ids

  # 토큰 ID를 텍스트로 변경
  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) # 지정된 구두점 문자 앞의 공백 삭제
    return text

In [15]:
# SimpleTokenizerV2 사용
# 텍스트 샘플
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)
print()

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.



In [16]:
# SimpleTokenizerV2로 샘플 텍스트 토큰화
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

# 텍스트로 재변환
print(tokenizer.decode(tokenizer.encode(text)))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


BOS(beginning of sequence): 텍스트의 시작 위치 표시

EOS(end of sequence): 텍스트의 끝에 위치. <|endoftext|>와 비슷하게 관련이 없는
여러 개의 텍스트 연결 시 유용

PAD(padding): 하나 이상의 배치 크기로 LLM 훈련 시 배치 안에 길이가 다른 텍스트에 포함 가능. 모든 텍스트의 길이를 동이라헤 맞추기 위해 PAD 사용 -> 배치에서 가장 긴 텍스트의 길이까지 확장 / 패딩

## SECTION 2.5 바이트 페어 인코딩

**tiktoken**: 러스트 언어. 매우 효율적으로 구현한 BPE 알고리즘 제공

**BPE(Byte Pair Encoding) 알고리즘**: 자주 같이 나오는 문자(또는 토큰) 쌍을 반복적으로 합쳐서 어휘를 만드는 서브워드(subword) 토크나이저 알고리즘

In [17]:
# tiktoken 버전 확인

from importlib.metadata import version
import tiktoken
print("tiktoken 버전:", version("tiktoken"))

tiktoken 버전: 0.12.0


In [18]:
# tictoken 라이브러리에서 BPE 토크나이저 초기화
tokenizer = tiktoken.get_encoding("gpt2")

In [19]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    " of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

strings = tokenizer.decode(integers)
print(strings)

# print(tokenizer.decode([30]))

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


1.  ` <|endoftext|>` 토큰은 50256과 같이 비교적 큰 토큰 ID에 할당
2.   BPE 토크나이저는 someynknownPlace와 같은 알지 못하는 단어를 정확히 인코딩 & 디코딩

- BPE 알고리즘
  - 어휘사전에 없는 단어를 더 작은 부분 단어, 심지어 개별 문자로 나누어 처음 본 단어 처리
  - 알지 못하는 단어 개별 문자로 분할 -> 토크나이저 & 이런 토크나이저로 훈련된 LLM이 훈련 데이터에 없는 단어 포함되더라도 모든 텍스트 처리 가능

## SECTION 2.6 슬라이딩 윈도로 데이터 샘플링하기

In [20]:
# BPE 토크나이저로 샘플 데이터 토큰화
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print("훈련 세트의 총 토큰 수:", len(enc_text))

훈련 세트의 총 토큰 수: 5145


In [21]:
# 데이터셋에 있는 처음 50개 토큰 삭제
enc_sample = enc_text[50:]

In [39]:
# 다음 단어 예측 작업 위해 입력-타깃 쌍 만드는 가장 쉽고 직관적 방법 중 하나
#   입력 토큰 담은 x, 입력에서 토큰 하나만큼 이동한 타깃 담은 y 변수 생성

context_size = 4    # 문맥의 크기: 입력에 얼마나 많은 토큰 포함할지 결정
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [23]:
# 다음 단어 예측 작업 구성
# 화살표 왼쪽 값: LLM이 받을 입력
# 화살표 오른쪽 토큰 ID: LLM이 예측해야 할 타깃 토큰 ID
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(context, "--->", desired)

[290] ---> 4920
[290, 4920] ---> 2241
[290, 4920, 2241] ---> 287
[290, 4920, 2241, 287] ---> 257


In [24]:
# 위의 코드에서 토큰 ID를 텍스트로 바꾸도록
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(tokenizer.decode(context), "--->", tokenizer.decode([desired]))

 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a


In [25]:
# 배치 입력과 타깃을 위한 데이터셋

import torch
from torch.utils.data import Dataset, DataLoader

# 파이토치 Dataset 클래스 기반, 데이터셋에서 개별 행 추출 방법 정의
# 각 행: input_chunk 텐서에 할당된 max_length만큼 여러 개의 토큰 ID 구성
  # target_chunk 텐서: 각 행에 상응하는 타깃 가짐
class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt)   # 전체 텍스트 토큰화

    # 슬라이딩 윈도 사용해 책을 max_length 길이의 중첩된 시퀀스로 나눔
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i+max_length]
      target_chunk = token_ids[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):    # 데이터셋에 있는 전체 행 수 반환
    return len(self.input_ids)

  def __getitem__(self, index):   # 데이터셋에서 하나의 행 반환
    return self.input_ids[index], self.target_ids[index]

In [26]:
# 입력-타깃 쌍의 배치 생성 위한 데이터 로더
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
  tokenizer = tiktoken.get_encoding("gpt2")   # 토크나이저 초기화
  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)  # 데이터셋 생성
  dataloader = DataLoader(
      dataset,
      batch_size = batch_size,
      shuffle=shuffle,
      drop_last = drop_last,    # True 지정 시 batch_size보다 작을 경우 훈련 손실 급격한 증가 피하기 위해 마지막 배치 삭제
      num_workers=num_workers   # 전처리에 사용할 CPU 프로세서 개수
  )

  return dataloader

In [27]:
# 문맥 크기 4, 배치 크기 1로 dataloader 테스트
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)    # 데이터로더를 파이썬 반복자(terator)로 변환 후 파이썬 내장 next() 함수로 다음 원소 추출

# 첫번째 텐서: 입력 토큰 ID 저장
# 두번째 텐서: 타깃 토큰 ID 저장
# max_length=4: 두 텐서는 4개의 토큰 ID 가짐
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [28]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [40]:
"""연습문제 2.2"""

# max_length=8, stride=2
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

ex_dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=8, stride=2, shuffle=False
)

ex_data_iter = iter(ex_dataloader)    # 데이터로더를 파이썬 반복자(terator)로 변환 후 파이썬 내장 next() 함수로 다음 원소 추출

# 첫번째 텐서: 입력 토큰 ID 저장
# 두번째 텐서: 타깃 토큰 ID 저장
# max_length=4: 두 텐서는 4개의 토큰 ID 가짐
ex_batch = next(ex_data_iter)
print(ex_batch)

[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]


![2.6_1](img/2.6_1.png)

In [30]:
# 배치 크기가 1보다 클 경우 데이터 로더로 샘플링
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4,
    shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("입력:\n", inputs)
print("\n타깃:\n", targets)

입력:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

타깃:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## SECTION 2.7 토큰 임베딩 만들기

![nn](img/2.7_1.png)
- 준비 과정: 텍스트 토큰화, 텍스트 토큰을 토큰 ID로 변경, 토큰 ID를 임베딩 벡터로 변경 작업 포함

In [31]:
# 입력 토큰 4개 가정
input_ids = torch.tensor([2, 3, 5, 1])

# 6개의 단어로 구성된 작은 어휘사전 가정, 크기 3인 임베딩
vocab_size = 6
output_dim = 3

# 결과 재현 가능하도록 랜덤 시드 123으로 지정
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)   # 임베딩 층에 있는 가중치 행렬 출력

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [32]:
# 임베딩 층의 가중치 행렬: 작고 랜덤한 값 가짐
  # LLM 최적화의 일부로 LLM 훈련 과정에서 최적화
  # 행 6개, 열 3개 => 어휘 사전에 있는 6개의 토큰 각각에 하나의 행 할당, 3개의 임베딩 차원 각각에 하나의 열 할당
# 이를 토큰 ID에 적용 -> 임베딩 벡터 생성
print(embedding_layer(torch.tensor([3])))

# 토큰 ID 3의 임베딩 벡터 값 == 이전의 4번째 행과 값
# 임베딩 층 = 토큰 ID 기반 가중치 행렬에서 행 추출하는 검색 연산 수행

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [41]:
# 하나의 토큰 ID를 3차원 임베딩 벡터로 바꾸는 방법
print(embedding_layer(input_ids))

# 가중치 행렬의 2, 3, 5, 1 열 출력

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


![nn](img/2.7_2.png)

## SECTION 2.8 단어 위치 인코딩하기

- LLM의 셀프 어텐션 매커니즘: 위치 정보x
  - -> LLM에 추가적인 위치 정보 주입
  - 상대 위치 임베딩 / 절대 위치 임베딩

- 절대 위치 임베딩
    - 시퀀스의 특정 위치에 직접 연관
    - 입력 시퀀스의 각 위치에 대해 고유한 임베딩 + 토큰 임베딩 => 정확한 위치 정보 추가
    - ![2.8_1](img/2.8_1.png)

- 상대 위치 임베딩
  - 토큰의 절대 위치에 초점X, 상대 위치 or 토큰 사이 거리 강조
  - 모델의 정확한 위치 아닌 멀미 떨어져 있는 정도 바탕으로 관계 학습
  - 길이가 다른 시퀀스에서도 더 잘 일반화 가능



In [34]:
# 초깃값으로 채워진 위치 임베딩 만들어 LLM 입력 준비
vocab_size = 50257
output_dim = 256
toekn_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [35]:
# token_embedding_layer 사용해 데이터 로더 통해 각 배치에 있는 토큰 ID를 256차원 벡터로 임베딩
# 데이터 로더 초기화

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, target = next(data_iter)
print("토큰 ID:\n", inputs)
print("\n입력 크기:\n", inputs.shape)   # 토큰 ID 텐서 차원 8x4 => 배치에 4개의 토큰 가진 텍스트 샘플 8개

토큰 ID:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

입력 크기:
 torch.Size([8, 4])


In [36]:
# 임베딩 층 사용해 이 토큰 ID 256차원 벡터로 임베딩
token_embeddings = toekn_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [37]:
# GPT 모델 임베딩 방법

context_length = max_length
# pos_Embeddings의 입력: 일반적으로 placeholder 벡터인 torch.aragne(context_length)
  # torch.arange(context_length): 숫자 0, 1, ..., 최대 입력 길이-1까지의 시퀀스
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [38]:
# 4x256 차원 임베딩 벡터에 pos_embeddings 텐서를 배치에 있는 4x256차원의 토큰 임베딩 텐서 8개에 각각 더함
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
